In [ ]:
# IMPORT LIBRARY
import numpy as np
import pandas as pd
import pyarrow as pa
from pathlib import Path


# LOAD & CLEAN

## LOAD

In [ ]:
files = [
    "data/yellow_tripdata_2024-01.parquet",
    "data/yellow_tripdata_2024-02.parquet",
    "data/yellow_tripdata_2024-03.parquet"
]

df_list = [pd.read_parquet(file) for file in files]

df = pd.concat(df_list, ignore_index=True)
original_len = len(df)

## CLEAN & VALIDATE

In [ ]:
df_filtered = df[
    (df["payment_type"] == 1) &
    (df["fare_amount"] > 0) &
    (df["trip_distance"] > 0) &
    (df["tip_amount"] >= 0) &
    (df["tpep_dropoff_datetime"] > df["tpep_pickup_datetime"])
].copy()
del df


In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df_filtered.isna().sum(),
    "missing_pct": df_filtered.isna().mean() * 100
}).sort_values("missing_count", ascending=False)

missing_summary

In [ ]:
duplicate_count = df_filtered.duplicated().sum()

print("Number of exact duplicate rows:", duplicate_count)
print(f"Duplicate percentage: {duplicate_count / len(df_filtered) * 100:.4f}%")

if duplicate_count > 0:
    df_filtered = df_filtered.drop_duplicates()

In [ ]:
validation_checks = {
    "non_credit_card": (df_filtered["payment_type"] != 1).sum(),
    "fare_amount <= 0": (df_filtered["fare_amount"] <= 0).sum(),
    "trip_distance <= 0": (df_filtered["trip_distance"] <= 0).sum(),
    "tip_amount < 0": (df_filtered["tip_amount"] < 0).sum(),
    "pickup_after_or_equal_dropoff": (
        df_filtered["tpep_pickup_datetime"] >= df_filtered["tpep_dropoff_datetime"]
    ).sum()
}

pd.Series(validation_checks)

In [ ]:
df_filtered["trip_duration_minutes"] = (
    df_filtered["tpep_dropoff_datetime"] - df_filtered["tpep_pickup_datetime"]
).dt.total_seconds() / 60

In [ ]:
df_filtered["average_speed_mph"] = (
    df_filtered["trip_distance"] / (df_filtered["trip_duration_minutes"] / 60)
)

In [ ]:
numeric_cols = [
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "tolls_amount",
    "total_amount",
    "congestion_surcharge",
    "Airport_fee",
    "trip_duration_minutes",
    "average_speed_mph"
]

df_filtered[numeric_cols].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
).T

### Filter outliers

In [ ]:
outlier_cols = [
    "trip_distance",
    "fare_amount",
    "trip_duration_minutes",
    "average_speed_mph"
]

lower_bounds = df_filtered[outlier_cols].quantile(0.025)
upper_bounds = df_filtered[outlier_cols].quantile(0.975)

bounds_table = pd.DataFrame({
    "lower_5pct": lower_bounds,
    "upper_95pct": upper_bounds
})

bounds_table

rows_before = len(df_filtered)

mask = pd.Series(True, index=df_filtered.index)

for col in outlier_cols:
    mask &= df_filtered[col].between(lower_bounds[col], upper_bounds[col])

df_cleaned = df_filtered[mask].copy()

rows_after = len(df_cleaned)

print("Rows before filtering:", rows_before)
print("Rows after filtering:", rows_after)
print("Rows removed:", rows_before - rows_after)
print(f"Removed percentage: {(rows_before - rows_after) / rows_before * 100:.2f}%")
del df_filtered

In [ ]:
validation_checks = {
    "duration <= 0": (df_cleaned["trip_duration_minutes"] <= 0).sum(),
    "duration > 180": (df_cleaned["trip_duration_minutes"] > 180).sum(),
    "trip_distance < 0.1": (df_cleaned["trip_distance"] < 0.1).sum(),
    "trip_distance > 100": (df_cleaned["trip_distance"] > 100).sum(),
    "fare_amount < 2.5": (df_cleaned["fare_amount"] < 2.5).sum(),
    "fare_amount > 500": (df_cleaned["fare_amount"] > 500).sum(),
    "avg_speed < 1": (df_cleaned["average_speed_mph"] < 1).sum(),
    "avg_speed > 80": (df_cleaned["average_speed_mph"] > 80).sum(),
    "avg_speed is infinite": np.isinf(df_cleaned["average_speed_mph"]).sum(),
    "avg_speed is missing": df_cleaned["average_speed_mph"].isna().sum()
}

pd.Series(validation_checks)

In [ ]:
validation_summary = {
    "rows": len(df_cleaned),
    "missing_cells": df_cleaned.isna().sum().sum(),
    "duplicate_rows": df_cleaned.duplicated().sum(),
    "non_credit_card_rows": (df_cleaned["payment_type"] != 1).sum(),
    "fare_amount <= 0": (df_cleaned["fare_amount"] <= 0).sum(),
    "trip_distance <= 0": (df_cleaned["trip_distance"] <= 0).sum(),
    "tip_amount < 0": (df_cleaned["tip_amount"] < 0).sum(),
    "dropoff <= pickup": (
        df_cleaned["tpep_dropoff_datetime"] <= df_cleaned["tpep_pickup_datetime"]
    ).sum(),
    "duration <= 0": (df_cleaned["trip_duration_minutes"] <= 0).sum(),
    "avg_speed <= 0": (df_cleaned["average_speed_mph"] <= 0).sum(),
}

pd.Series(validation_summary)

In [ ]:
assert (df_cleaned["fare_amount"] > 0).all(), "There are still rows with fare_amount <= 0"

df_cleaned["tip_percentage"] = df_cleaned["tip_amount"] / df_cleaned["fare_amount"]

df_cleaned["high_tip"] = (df_cleaned["tip_percentage"] >= 0.20).astype(int)

In [ ]:
df_cleaned["tip_percentage"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
).T

In [ ]:
target_summary = pd.DataFrame({
    "count": df_cleaned["high_tip"].value_counts(),
    "percentage": df_cleaned["high_tip"].value_counts(normalize=True) * 100
})

target_summary

In [ ]:
working_dir = Path.cwd()
file_name = "data/clean_yellow_taxi_2024_q1.parquet"

save_path = working_dir / file_name

# Save parquet file
df_cleaned.to_parquet(save_path, index=False)

print("File exists:", save_path.exists())